# dietary-dti-screen — Colab quickstart

Thin wrapper over the [`dti-screen`](https://github.com/anindya2710/dietary-dti-screen) package. All logic lives in the repo and is
covered by its test suite; this notebook just installs it and calls the same entry point
the CLI uses, so there is no second copy of the pipeline to drift out of sync.

**Change `REPO_URL` below to your fork before running.**

What it does: validate a SMILES library → fetch target sequences from UniProt → score
with a 4-model DeepPurpose ensemble → benchmark against known actives/decoys → write one
ranked CSV per target.

Read `README.md` in the repo for how to interpret the output. The short version: check the
control-benchmark AUC **before** reading your top hits. An AUC near 0.5 means that
target's rankings are noise.

## 1. Install

Order matters, and three things here are deliberate:

- **`--no-deps` on DeepPurpose** — its declared requirements pull `ax-platform` and
  `dgllife`, which this code path never uses (~160 extra packages).
- **`descriptastorus` from git** — a hard top-level import of `DeepPurpose.utils` that
  is not on PyPI and is missing from DeepPurpose's own `requirements.txt`. Without it,
  `import DeepPurpose` fails outright.
- **No `dgl`** — not needed, because the MPNN encoder is deliberately excluded.

Takes 2–4 minutes.

In [ ]:
REPO_URL = "https://github.com/anindya2710/dietary-dti-screen.git"   # <-- point this at your fork
BRANCH = "main"

# 1. DeepPurpose itself, without its over-broad declared dependencies.
!pip install -q --no-deps DeepPurpose==0.1.5

# 2. This package + its real runtime deps (incl. descriptastorus from git).
#    torch>=1.13 and rdkit are satisfied by Colab's preinstalled builds where possible,
#    so pip leaves them alone rather than reinstalling a CUDA stack.
!pip install -q "git+{REPO_URL}@{BRANCH}"

print("\nInstall finished.")

### Restart the runtime now

`lifelines` (a module-level import in `DeepPurpose/DTI.py`) pins `pandas<3.0`, so pip
downgrades Colab's pandas during the step above. Colab has already imported the old
module objects, which produces confusing `AttributeError`s rather than a clean failure.

**Runtime → Restart session**, then continue from the next cell. Do not re-run the
install cell — the packages survive a restart.

In [ ]:
import dti_screen
from dti_screen import ScreenConfig, run_screen

print("dti_screen", dti_screen.__version__)

import torch, pandas as pd, numpy as np
from rdkit import Chem
print("torch   ", torch.__version__)
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)
print("rdkit   ", Chem.rdBase.rdkitVersion)
print("CUDA    ", torch.cuda.is_available(), "(a GPU barely helps here - see README)")

## 2. Verify the control set

Fast, no downloads. Every control structure is checked against its literature molecular
formula; this is **fatal** on mismatch, because an unverified control set cannot support
the benchmark built on it.

In [ ]:
from dti_screen import verify_control_structures

controls = verify_control_structures()
print(f"{len(controls)} control structures verified against expected formulas.\n")
print(controls.groupby("control_for").size().to_string())
controls.head(20)

## 3. Provide your library

Upload a CSV with a SMILES column using the file browser (folder icon, left sidebar), or
mount Drive. Set `SMILES_COL` to whatever your file actually calls it — FooDB exports use
`moldb_smiles`, not `SMILES`.

Leave `INPUT_CSV` as-is to use the 61-molecule verified example set that ships with the
repo.

In [ ]:
import os, glob
from importlib.resources import files

# Default: the example set bundled with the installed package.
try:
    INPUT_CSV = str(files("dti_screen").joinpath("../../data/example_ingredients.csv").resolve())
except Exception:
    INPUT_CSV = ""

# Prefer anything you uploaded to the Colab working directory.
uploaded = [p for p in glob.glob("*.csv") if "prediction" not in p.lower()]
if uploaded:
    INPUT_CSV = uploaded[0]
    print(f"Using uploaded file: {INPUT_CSV}")
elif os.path.exists(INPUT_CSV):
    print(f"Using the bundled example set: {INPUT_CSV}")
else:
    # Fall back to fetching the example CSV straight from the repo.
    !wget -q -O example_ingredients.csv https://raw.githubusercontent.com/anindya2710/dietary-dti-screen/main/data/example_ingredients.csv
    INPUT_CSV = "example_ingredients.csv"
    print(f"Fetched the example set from GitHub: {INPUT_CSV}")

SMILES_COL = "SMILES"     # e.g. "moldb_smiles" for FooDB exports
NAME_COL   = "name"       # or None to auto-generate CMPD_00001 style IDs

import pandas as pd
preview = pd.read_csv(INPUT_CSV)
print(f"\n{len(preview):,} rows, columns = {list(preview.columns)}")
assert SMILES_COL in preview.columns, (
    f"{SMILES_COL!r} not in {list(preview.columns)} - set SMILES_COL to the right column"
)
preview.head()

## 4. Run the screen

~1 minute per 1,000 molecules per target on CPU. One failing target, checkpoint or model
degrades the run rather than ending it.

In [ ]:
cfg = ScreenConfig(
    input_csv=INPUT_CSV,
    smiles_col=SMILES_COL,
    name_col=NAME_COL,
    output_dir="predictions",
    work_dir="dp_work",
    include_controls=True,      # this is what makes the output interpretable
    aggregation="agg_mean_max", # DeepPurpose oneliner's default; also "mean"/"max_effect"
)

out = run_screen(cfg)

## 5. Results

`predicted_Kd_nM` is ascending, so **lower is stronger**. The equivalent `pKd_aggregate`
is descending (higher is stronger). `pKd_std` is ensemble disagreement — large values mean
the four models do not agree and the consensus is thin.

In [ ]:
for label, res in out["results"].items():
    print(f"\n=== {label}: top 10 library compounds (controls excluded) ===")
    view = res[~res["is_control"]].head(10)
    if view.empty:
        print("  (no non-control compounds)")
        continue
    print(view[["rank", "compound_name", "predicted_Kd_nM", "pKd_aggregate", "pKd_std"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

## 6. The control benchmark — read this before the hits above

For each target: does the ensemble rank that target's **known actives** above the
**dietary decoys**? `AUC` is the probability a randomly chosen active outranks a randomly
chosen decoy.

| AUC | Meaning |
|-----|---------|
| ≥ 0.8 | Actives clearly separated. The ranking carries signal. |
| 0.6–0.8 | Weak separation. A soft prior at best. |
| < 0.6 | No usable separation. **This target's scores are noise.** |

An AUC near 0.5 is a legitimate, reportable result — not a bug, and not something to work
around. It is also the most valuable thing this notebook produces.

In [ ]:
bench = out["benchmark"]
display(bench)

print()
for _, r in bench.iterrows():
    print(f"  {r['target']:10s} AUC = {r['auc']:.3f}   {r['verdict']}")

## 7. Save your results

Colab's filesystem is **ephemeral** — your uploaded library and these CSVs disappear when
the runtime disconnects. Download them now, or mount Drive and copy them across.

In [ ]:
import shutil, os
from google.colab import files as colab_files

archive = shutil.make_archive("dti_predictions", "zip", "predictions")
print(f"{archive}  ({os.path.getsize(archive)/1e6:.2f} MB)")
print("Contents:", os.listdir("predictions"))

colab_files.download(archive)

# Or, to keep them in Drive instead:
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copytree("predictions", "/content/drive/MyDrive/dti_predictions", dirs_exist_ok=True)

## What you can and cannot claim

**Supported:** a reproducible pipeline; a ranking under one stated model and aggregation
rule; a quantitative statement about whether the method separates actives from decoys.

**Not supported:**

- That any top-ranked molecule binds its target — these are out-of-domain predictions
  from sequence-only models, and the absolute pKd/nM values should not be quoted as
  affinities.
- Any physiological or health claim. Target engagement is not a dietary effect:
  bioavailability, first-pass metabolism, achievable plasma concentration from realistic
  intake and BBB penetration all sit between these numbers and anything happening in a
  person.
- Cross-target rank comparisons — each target's scores are separately calibrated.

See the repo README for the full reasoning and for next steps worth doing.